# ML Zoomcamp 2026 — Homework 2 (Machine Learning for Regression)

Alvo: `fuel_efficiency_mpg`. Colunas usadas: `engine_displacement`, `horsepower`, `vehicle_weight`, `model_year`.

In [ ]:
import pandas as pd
import numpy as np

URL = (
    "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/"
    "main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
)
cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = pd.read_csv(URL)[cols].astype(float)
df.head()

In [ ]:
def split(seed):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)
    return (df.iloc[idx[:n_train]],
            df.iloc[idx[n_train:n_train + n_val]],
            df.iloc[idx[n_train + n_val:]])


def prep(d, fill):
    d = d.copy()
    d['horsepower'] = d['horsepower'].fillna(fill)
    X = d[['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']].values
    return X, d['fuel_efficiency_mpg'].values


def train(X, y, r=0.0):
    X = np.column_stack([np.ones(X.shape[0]), X])
    XTX = X.T.dot(X) + r * np.eye(X.shape[1])
    w = np.linalg.solve(XTX, X.T.dot(y))
    return w[0], w[1:]


def rmse(y, yp):
    return np.sqrt(((y - yp) ** 2).mean())


tr, va, te = split(42)

## Q1. Coluna com valores faltantes — **horsepower** (877 nulos)

In [ ]:
na = df.isna().sum()
na[na > 0]

## Q2. Mediana de horsepower — **254**

In [ ]:
df['horsepower'].median()

## Q3. Preencher com 0 ou com a média — **With mean**

RMSE com 0 = 2.205; RMSE com a média do treino = 2.202.

In [ ]:
res = {}
for label, fill in [('0', 0.0), ('mean', tr['horsepower'].mean())]:
    Xtr, ytr = prep(tr, fill)
    Xva, yva = prep(va, fill)
    w0, w = train(Xtr, ytr)
    res[label] = round(rmse(yva, w0 + Xva.dot(w)), 3)
res

## Q4. Melhor `r` da regressão regularizada — **0**

`{0: 2.2053, 0.01: 2.2058, 0.1: 2.2241, 1: 2.3492, 5: 2.4094, 10: 2.4195, 100: 2.4292}`

In [ ]:
Xtr, ytr = prep(tr, 0.0)
Xva, yva = prep(va, 0.0)
q4 = {}
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train(Xtr, ytr, r)
    q4[r] = round(rmse(yva, w0 + Xva.dot(w)), 4)
q4, min(q4, key=lambda k: (q4[k], k))

## Q5. Desvio padrão dos RMSE nas seeds 0–9 — **0.029**

In [ ]:
scores = []
for s in range(10):
    t, v, _ = split(s)
    Xt, yt = prep(t, 0.0)
    Xv, yv = prep(v, 0.0)
    w0, w = train(Xt, yt)
    scores.append(rmse(yv, w0 + Xv.dot(w)))
round(float(np.std(scores)), 3)

## Q6. RMSE no teste (seed 9, train+val, r=0.001) — **2.236**

In [ ]:
t, v, te = split(9)
full = pd.concat([t, v])
Xf, yf = prep(full, 0.0)
Xte, yte = prep(te, 0.0)
w0, w = train(Xf, yf, 0.001)
rmse(yte, w0 + Xte.dot(w))